In [13]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from genkit._noise import sample_scaled_isotropic_alpha_stable
from genkit.diffusion import DDPMV, DLPMEps
from genkit.metrics import mmd_rbf
from genkit.nn import MLPModel
from genkit.training import train


COLORS = {"DDPM": "#2f6fbb", "DLPM": "#d9792b"}
DEVICE = torch.device("cpu")
DTYPE = torch.float32
SEED = 0
N_TRIALS = 3

ALPHA = 1.7
DIMS = [5, 30, 50]
N_TRAIN_GRID = np.array([10, 30, 100, 300, 500, 1_000, 2_000, 3_000, 5_000, 10_000], dtype=int)
SUMMARY_N_TRAIN = np.array([10, 100, 1_000, 10_000], dtype=int)
N_TEST = N_MMD = 2_000
SAMPLE_CHUNK_SIZE = 500
BASELINE_NAME = "test-vs-test"

N_REVERSE_STEPS = 64
DDPM_N_STEPS = N_REVERSE_STEPS
DLPM_N_STEPS = N_REVERSE_STEPS + 1  # DLPM's native sampler runs n_steps - 1 reverse updates.
DDPM_SIGMA_MAX = 5.0

N_EPOCHS = 128
BATCH_SIZE = 256
LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
DEPTH = 3
WIDTH = 128
TIME_DIM = 64

TRAIN_BASE_KWARGS = dict(
    batch_size=BATCH_SIZE,
    lr=LR,
    device=DEVICE,
    use_adamw=True,
    weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM,
    lr_schedule="cosine",
    cosine_eta_min_ratio=0.05,
    freq_logging=0,
)


In [17]:
def sample_alpha_stable(n_samples, dim):
    return sample_scaled_isotropic_alpha_stable(n_samples, dim=int(dim), alpha=ALPHA, device=DEVICE, dtype=DTYPE)


def make_net(dim):
    return MLPModel(input_dim=int(dim), output_dim=int(dim), width=WIDTH, depth=DEPTH, time_dim=TIME_DIM).to(device=DEVICE, dtype=DTYPE)


def set_seed(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))


def train_model(model, x_train):
    train_kwargs = dict(TRAIN_BASE_KWARGS, n_epochs=N_EPOCHS, stats_freq_epochs=N_EPOCHS)
    return train(model, x_train, **train_kwargs)


@torch.no_grad()
def evaluate_samples(dim, name, trial, n_train, x_ref, x_gen):
    x_ref = x_ref.detach().cpu().to(torch.float64)
    x_gen = x_gen.detach().cpu().to(torch.float64)
    return {"dim": int(dim), "trial": trial, "n_train": n_train, "model": name, "mmd_rbf": mmd_rbf(x_ref[:N_MMD], x_gen[:N_MMD])}


def evaluate_sampler(dim, name, sample_fn, trial, n_train, x_test):
    chunks = []
    for start in range(0, len(x_test), SAMPLE_CHUNK_SIZE):
        size = min(SAMPLE_CHUNK_SIZE, len(x_test) - start)
        chunks.append(sample_fn(size).detach().cpu())
    return evaluate_samples(dim, name, trial, n_train, x_test, torch.cat(chunks, dim=0))


def summary_values(dim, curve_table):
    available = set(curve_table.loc[curve_table["dim"] == dim, "n_train"].astype(int))
    return [int(n) for n in SUMMARY_N_TRAIN if int(n) in available]


def display_summary(dim, curve_table, baseline_by_dim):
    summary_n = summary_values(dim, curve_table)
    summary = curve_table[(curve_table["dim"] == dim) & curve_table["n_train"].isin(summary_n)]
    summary = summary.pivot(index="n_train", columns="model", values="mmd_rbf_mean").reindex(summary_n)
    baseline = float(baseline_by_dim.get(dim, np.nan))

    lines = [f"**d = {dim}**; test-vs-test MMD-RBF = {baseline:.3e}", "", "| nb samples | DDPM | DLPM |", "|---:|---:|---:|"]
    for n_train, row in summary.iterrows():
        lines.append(f"| {int(n_train):,} | {row.get('DDPM', np.nan):.3e} | {row.get('DLPM', np.nan):.3e} |")
    raw_table = "\n".join(lines)
    return raw_table, Markdown(raw_table)


def plot_curve(dim, curve_table, colors=COLORS):
    xticks = summary_values(dim, curve_table)
    fig, ax = plt.subplots(figsize=(3.2, 2.8))

    for model in ["DDPM", "DLPM"]:

        model_curve = curve_table[(curve_table["dim"] == dim) & (curve_table["model"] == model)].sort_values("n_train")
        x = model_curve["n_train"].to_numpy(dtype=float)
        y = model_curve["mmd_rbf_mean"].to_numpy(dtype=float)
        marker_idx = [i for i, n in enumerate(x.astype(int)) if n in set(xticks)]

        ax.plot(x, y, marker="o", markevery=marker_idx, markersize=4.5, linewidth=3.0, color=colors[model], label=model)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Nb samples")
    ax.set_ylabel(f"MMD-RBF (dim={dim})")
    ax.set_xticks(xticks, [f"{int(n):,}" for n in xticks], rotation=20, ha="right")
    ax.grid(axis="both", which="both", alpha=0.25)
    ax.legend(frameon=False, fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    plt.show()


In [3]:
rows = []
for dim in DIMS:
    print(f"d={dim}")
    set_seed(SEED + 100_000 * int(dim))
    x_test = sample_alpha_stable(N_TEST, dim)

    for trial in range(1, N_TRIALS + 1):
        print(f"  trial {trial} / {N_TRIALS}:", end="")
        trial_seed = SEED + 100_000 * int(dim) + 10_000 * trial
        set_seed(trial_seed)
        x_pool = sample_alpha_stable(int(N_TRAIN_GRID.max()), dim)
        perm = torch.randperm(len(x_pool), device=x_pool.device)

        for i, n_train in enumerate(N_TRAIN_GRID):
            n_train = int(n_train)
            x_train = x_pool.index_select(0, perm[:n_train])
            point_seed = trial_seed + 100 * i
            print(f" n={n_train}:", end="")

            start = time.time()
            set_seed(point_seed + 1)
            ddpm = DDPMV(net=make_net(dim), dim=dim, n_steps=DDPM_N_STEPS, sigma_max=DDPM_SIGMA_MAX, sampler="ddpm", device=DEVICE, fdtype=DTYPE)
            ddpm, _ = train_model(ddpm, x_train)
            set_seed(point_seed + 2)
            rows.append(evaluate_sampler(dim, "DDPM", ddpm.sample, trial, n_train, x_test))
            print(f" DDPM {time.time() - start:.1f}s", end="")

            start = time.time()
            set_seed(point_seed + 1)
            dlpm = DLPMEps(net=make_net(dim), dim=dim, n_steps=DLPM_N_STEPS, alpha=ALPHA, device=DEVICE, fdtype=DTYPE)
            dlpm, _ = train_model(dlpm, x_train)
            set_seed(point_seed + 2)
            rows.append(evaluate_sampler(dim, "DLPM", dlpm.sample, trial, n_train, x_test))
            print(f" DLPM {time.time() - start:.1f}s", end="")

        start = time.time()
        set_seed(trial_seed + 999)
        rows.append(evaluate_samples(dim, BASELINE_NAME, trial, None, x_test, sample_alpha_stable(len(x_test), dim)))
        print(f" {BASELINE_NAME} {time.time() - start:.1f}s")


d=5
  trial 1 / 3: n=10: DDPM 1.8s DLPM 1.3s n=30: DDPM 1.3s DLPM 1.3s n=100: DDPM 1.1s DLPM 1.4s n=300: DDPM 2.1s DLPM 2.4s n=500: DDPM 2.1s DLPM 2.3s n=1000: DDPM 3.9s DLPM 5.1s n=2000: DDPM 10.0s DLPM 8.1s n=3000: DDPM 13.5s DLPM 15.6s n=5000: DDPM 15.8s DLPM 22.3s n=10000: DDPM 44.1s DLPM 48.0s test-vs-test 0.5s
  trial 2 / 3: n=10: DDPM 4.4s DLPM 1.8s n=30: DDPM 2.7s DLPM 1.7s n=100: DDPM 1.4s DLPM 1.7s n=300: DDPM 2.3s DLPM 2.5s n=500: DDPM 2.7s DLPM 2.7s n=1000: DDPM 4.2s DLPM 4.4s n=2000: DDPM 9.7s DLPM 7.5s n=3000: DDPM 10.3s DLPM 14.3s n=5000: DDPM 18.0s DLPM 24.1s n=10000: DDPM 36.7s DLPM 41.9s test-vs-test 0.4s
  trial 3 / 3: n=10: DDPM 1.3s DLPM 1.6s n=30: DDPM 1.8s DLPM 2.6s n=100: DDPM 1.7s DLPM 1.7s n=300: DDPM 2.2s DLPM 2.5s n=500: DDPM 5.4s DLPM 3.0s n=1000: DDPM 4.1s DLPM 4.3s n=2000: DDPM 6.9s DLPM 7.8s n=3000: DDPM 15.4s DLPM 13.5s n=5000: DDPM 21.5s DLPM 20.3s n=10000: DDPM 39.3s DLPM 46.5s test-vs-test 0.5s
d=30
  trial 1 / 3: n=10: DDPM 1.5s DLPM 1.7s n=30: DDPM

In [27]:
results = pd.DataFrame(rows)
curve_results = results[results["model"] != BASELINE_NAME].copy()
curve_results["n_train"] = curve_results["n_train"].astype(int)

curve_table = curve_results.groupby(["dim", "model", "n_train"], sort=True)["mmd_rbf"].agg(["mean", "std"]).fillna(0.0)
curve_table = curve_table.rename(columns={"mean": "mmd_rbf_mean", "std": "mmd_rbf_std"}).reset_index()

model_order = ["DDPM", "DLPM"]
dims = [int(dim) for dim in DIMS]
sample_grid = [int(n) for n in SUMMARY_N_TRAIN]

big_table = curve_table[curve_table["model"].isin(model_order)]
big_table = big_table.pivot(index="n_train", columns=["dim", "model"], values="mmd_rbf_mean")
big_table = big_table.reindex(index=sample_grid)

columns = pd.MultiIndex.from_product([dims, model_order], names=["dim", "model"])
big_table = big_table.reindex(columns=columns)

def fmt(value):
    return "NA" if pd.isna(value) else f"{float(value):.3e}"

headers = ["nb samples"] + [f"d={dim} {model}" for dim in dims for model in model_order]
align = ["---:"] * len(headers)

lines = [
    "| " + " | ".join(headers) + " |",
    "| " + " | ".join(align) + " |",
]

for n_train, row in big_table.iterrows():
    values = [f"{int(n_train):,}"]
    for dim in dims:
        for model in model_order:
            values.append(fmt(row[(dim, model)]))
    lines.append("| " + " | ".join(values) + " |")

markdown_table = "\n".join(lines)
print(markdown_table)
display(Markdown(markdown_table))

| nb samples | d=5 DDPM | d=5 DLPM | d=30 DDPM | d=30 DLPM | d=50 DDPM | d=50 DLPM |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 10 | 5.850e-02 | 3.119e-01 | 1.355e-01 | 2.812e-01 | 9.815e-02 | 2.871e-01 |
| 100 | 1.464e-02 | 1.833e-01 | 7.386e-02 | 2.638e-01 | 9.168e-02 | 2.458e-01 |
| 1,000 | 1.459e-03 | 2.136e-03 | 2.406e-02 | 2.604e-01 | 5.065e-02 | 2.537e-01 |
| 10,000 | 2.275e-03 | 6.656e-04 | 2.308e-03 | 1.033e-03 | 1.142e-03 | 1.984e-02 |


| nb samples | d=5 DDPM | d=5 DLPM | d=30 DDPM | d=30 DLPM | d=50 DDPM | d=50 DLPM |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 10 | 5.850e-02 | 3.119e-01 | 1.355e-01 | 2.812e-01 | 9.815e-02 | 2.871e-01 |
| 100 | 1.464e-02 | 1.833e-01 | 7.386e-02 | 2.638e-01 | 9.168e-02 | 2.458e-01 |
| 1,000 | 1.459e-03 | 2.136e-03 | 2.406e-02 | 2.604e-01 | 5.065e-02 | 2.537e-01 |
| 10,000 | 2.275e-03 | 6.656e-04 | 2.308e-03 | 1.033e-03 | 1.142e-03 | 1.984e-02 |